# Unique IDs for duplicated sequences (canonical and isoform)
For the duplicated sequences identified in the last otebook, we decided to create new identifiers merging together the identifiers of the duplicated sequences. This meant merging the IDs, protein names, gene names, protein existence and sequence variant. If an isoform sequence was a duplicate of a canonical sequence, the longer one of the canonical sequences was kept as the sequence of the new identifier. The merged information and the corresponding sequence were put into a new fasta file. This was also done for the one isoform sequence that was duplicated.
In this notebook I will remove all the not unique identifiers from the full proteome dataframe and replace them with our new unique identifiers.

# Setup
## Import Packages
We will use the a mapping from ifoforms to their canonical forms downloaded from UniProt to get a table containing only the canonical IDs and the Isoform IDs that exist for this canonical form.

In [32]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [33]:
import pandas as pd
from functions import read_fastafile

In [34]:
# Display session information
session_info.show()

In [35]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/01_UniProt"):
    print("WARNING: The working directory is not set to the '01_UniProt'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '01_Uniprot' folder (\"{cwd}\").")

Current working directory: /Users/lauranieba/Documents/Uni/Master_ETH/FS26/Lab_Rotation_Davos/Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/01_UniProt


In [36]:
# Data directories
fasta_dir = "data/raw_data/fasta"
duplicates_dir = "data/sequence_duplicates"
unique_dir = "data/unique"

## Read in fasta files and duplicates list

In [37]:
unique_canonical_duplicates = read_fastafile(os.path.join(duplicates_dir, 'unique_canonical_duplicates.fasta'))
unique_isoform_duplicates = read_fastafile(os.path.join(duplicates_dir, 'unique_isoform_duplicates.fasta'))

fasta_canonical = read_fastafile(os.path.join(fasta_dir, 'uniprotkb_canonical_2026_03_31.fasta'))
fasta_full_proteome = read_fastafile(os.path.join(fasta_dir, 'uniprotkb_full_proteome_2026_03_31.fasta'))
fasta_alternative_products = read_fastafile(os.path.join(fasta_dir, 'uniprotkb_alternative_products_isoforms_2026_03_31.fasta'))

duplicates_list = pd.read_csv(os.path.join(duplicates_dir, 'duplicates_list_01042026.csv'))

In [38]:
unique_canonical_duplicates.to_csv(os.path.join(duplicates_dir, 'unique_canonical_duplicates.csv'))

## Build unique dataframes 
We will remove the rows of duplicates from the canonical df and full proteome df. For Q9NZJ9-2 (mapped to A0A024RBG1_Q9NZJ9, canonical sequence of Q9NZJ9), Q96PT3-2 (mapped to O43812_Q96PT3, canonical sequence of Q96PT3) and P0DPK4-2 (mapped to Q7Z3S9_P0DPK4, canonical sequence of P0DPK4) we need to remove the isofomr ids of these and add the canonical base ids instead.

In [39]:
duplicate_ids = duplicates_list["ID"].tolist()

duplicate_ids.append('P0DPK4')
duplicate_ids.append('Q96PT3')
duplicate_ids.append('Q9NZJ9')

duplicate_ids.remove('P0DPK4-2')
duplicate_ids.remove('Q96PT3-2')
duplicate_ids.remove('Q9NZJ9-2')

canonical_no_duplicates = fasta_canonical[~fasta_canonical['ID'].isin(duplicate_ids)]
isoform_no_duplicates = fasta_alternative_products[~fasta_alternative_products['ID'].isin(duplicate_ids)]
full_proteome_no_duplicates = fasta_full_proteome[~fasta_full_proteome['ID'].isin(duplicate_ids)]

### Canonical

In [40]:
# stack new unique canonical ids and filtered canonical df (full proteome is done later when we create unique IDs for isoforms)
canonical_unique = pd.concat([canonical_no_duplicates, unique_canonical_duplicates], ignore_index=True)
canonical_unique

,ID,seqID,seq,len
0,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,515
1,A0A096LP01,sp|A0A096LP01|SIM26_HUMAN,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95
2,A0A0B4J2F0,sp|A0A0B4J2F0|PIOS1_HUMAN,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54
3,A0A0C5B5G6,sp|A0A0C5B5G6|MOTSC_HUMAN,MRWQEMGYIFYPRKLR,16
4,A0A0K2S4Q6,sp|A0A0K2S4Q6|CD3CH_HUMAN,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201
...,...,...,...,...
20345,Q5DJT8_P0DMV1_P0DMV2,sp|Q5DJT8_P0DMV1_P0DMV2|CT452_CT458_CT459_HUMAN,MTDKTEKVAVDPETVFKRPRECDSPSYQKRQRMALLARKQGAGDSL...,189
20346,Q9BYR9_P0C7H8,sp|Q9BYR9_P0C7H8|KRA24_KRA23_HUMAN,MTGSCCGSTLSSLSYGGGCCQPCCCRDPCCCRPVTCQTTVCRPVTC...,128
20347,A0A0A6YYL3_H3BUK9,sp|A0A0A6YYL3_H3BUK9|POTEB_POTB2_HUMAN,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGTSNVGTS...,544
20348,A0A075B759_P0DN26,sp|A0A075B759_P0DN26|PAL4E_PAL4F_HUMAN,MVNSVVFFEITRDGKPLGRISIKLFADKIPKTAENFRALSTGEKGF...,164


In [41]:
# check for ID uniqueness...
print(canonical_unique['ID'].is_unique) 

True


In [42]:
#...and sequence uniqueness
print(canonical_unique['seq'].is_unique)

True


### Isoform

In [43]:
# stack new unique isoform ids and filtered isoform df 
isoform_unique = pd.concat([isoform_no_duplicates, unique_isoform_duplicates], ignore_index=True)
isoform_unique

,ID,seqID,seq,len
0,A0A096LP01,sp|A0A096LP01|SIM26_HUMAN,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95
1,A0A0K2S4Q6,sp|A0A0K2S4Q6|CD3CH_HUMAN,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201
2,A0A1B0GTW7,sp|A0A1B0GTW7|CIROP_HUMAN,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,788
3,A0AV02,sp|A0AV02|S12A8_HUMAN,MTQMSQVQELFHEAAQQDALAQPQPWWKTQLFMWEPVLFGTWDGVF...,714
4,A0AV96,sp|A0AV96|RBM47_HUMAN,MTAEDSTAAMSSDSAAGSSAKVPEGVAGAPNEAALLALMERTGYSM...,593
...,...,...,...,...
32794,Q9Y6X0-2,sp|Q9Y6X0-2|SETBP_HUMAN,MESRETLSSSRQRGGESDFLPVSSAKPPAAPGCAGEPLLSTPGPGK...,242
32795,Q9Y6X4-2,sp|Q9Y6X4-2|F169A_HUMAN,MAFPVDMLENCSHEELENSAEDYMSDLRCGDPENPECFSLLNITIP...,127
32796,Q9Y6Y8-2,sp|Q9Y6Y8-2|S23IP_HUMAN,MAERKPNGGSGGASTSSSGTNLLFSSSATEFSFNVPFIPVTQASAS...,924
32797,X6R8D5-2,sp|X6R8D5-2|CMIP3_HUMAN,MGRKEHESPSQPHMCGWEDSQKPSVPSHGPKTPSCKGVKAPHSSRP...,127


In [44]:
# check for ID uniqueness...
print(isoform_unique['ID'].is_unique) 

True


In [45]:
#...and sequence uniqueness
print(isoform_unique['seq'].is_unique)

True


### Full Proteome

In [46]:
# stack unique canonical and isoform ids and filtered full proteome df 
full_proteome_unique = pd.concat([full_proteome_no_duplicates, unique_canonical_duplicates], ignore_index=True)
full_proteome_unique = pd.concat([full_proteome_unique, unique_isoform_duplicates], ignore_index=True)
full_proteome_unique

,ID,seqID,seq,len
0,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,515
1,A0A096LP01,sp|A0A096LP01|SIM26_HUMAN,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95
2,A0A0B4J2F0,sp|A0A0B4J2F0|PIOS1_HUMAN,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54
3,A0A0C5B5G6,sp|A0A0C5B5G6|MOTSC_HUMAN,MRWQEMGYIFYPRKLR,16
4,A0A0K2S4Q6,sp|A0A0K2S4Q6|CD3CH_HUMAN,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201
...,...,...,...,...
42475,Q9BYR9_P0C7H8,sp|Q9BYR9_P0C7H8|KRA24_KRA23_HUMAN,MTGSCCGSTLSSLSYGGGCCQPCCCRDPCCCRPVTCQTTVCRPVTC...,128
42476,A0A0A6YYL3_H3BUK9,sp|A0A0A6YYL3_H3BUK9|POTEB_POTB2_HUMAN,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGTSNVGTS...,544
42477,A0A075B759_P0DN26,sp|A0A075B759_P0DN26|PAL4E_PAL4F_HUMAN,MVNSVVFFEITRDGKPLGRISIKLFADKIPKTAENFRALSTGEKGF...,164
42478,E9PI22_P0DMB1,sp|E9PI22_P0DMB1|P23D1_P23D2_HUMAN,MYGYRRLRSPRDSQTEPQNDNEGETSLATTQMNPPKRRQVEQGPST...,279


In [47]:
# check for ID uniqueness...
print(full_proteome_unique['ID'].is_unique) 

True


In [48]:
#...and sequence uniqueness
print(full_proteome_unique['seq'].is_unique)

False


Finally we can export all the dataframes with unique IDs and sequences to use them for the mapping in the next notebook.

In [49]:
canonical_unique.to_csv(os.path.join(unique_dir, 'canonical_unique.csv'))
isoform_unique.to_csv(os.path.join(unique_dir, 'isoform_unique.csv'))
full_proteome_unique.to_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))